# ⛓️ Lesson 5: Advanced Chains & LangChain Expression Language (LCEL)

Welcome to Lesson 5! Up until now, we have used LangChain components somewhat independently. Today, we are diving deep into the core engine that binds everything together: **LCEL (LangChain Expression Language)**.

### 🎯 The Session Goal
Before we let an AI make its own decisions using an **Agent (Lesson 6)**, we must master how to build explicit, predictable, and robust pipelines. We will learn how data flows through the pipe (`|`) operator, how LangChain handles inputs under the hood, and how to branch our code logically.

### 🧠 The Core Concepts
1. **The Runnable Protocol**: Understanding why any LangChain component can seamlessly connect to another.
2. **Deterministic Layouts**: Building multi-step pipelines where the output of one LLM prompt becomes the direct input for the next prompt.
3. **Input/Output Schemas**: Visualizing exactly how data transforms as it moves through our pipeline.


In [7]:
# Install the required packages
!pip install -q langchain-core langchain-openai openai

import os
from google.colab import userdata

# Securely set your OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')


## 📑 Step 1: The Concept of Piping Data

To understand why LangChain uses the pipe operator (`|`), we need to look at how software engineers traditionally pass data between separate functions.

### The Problem with Standard Python Functions
When you want the output of `Function A` to become the input of `Function B`, your code can quickly become unreadable, cluttered with nested parentheses, or bogged down by temporary variables:

```python
# The hard-to-read nested way:
result = function_c(function_b(function_a(initial_input)))

# The messy temporary variable way:
step1 = function_a(initial_input)
step2 = function_b(step1)
result = function_c(step2)
```

### The Solution: The Pipe Operator (`|`)
LangChain borrows the pipe operator (`|`) from Unix/Linux terminal commands. It allows data to flow cleanly from **left to right**.

Let's look at a basic Python example to see how LangChain overrides this operator to create cleaner, more readable pipelines.


In [8]:
from langchain_core.runnables import RunnableLambda

# 1. Define three simple, independent logic steps
add_five = RunnableLambda(lambda x: x + 5)
multiply_by_two = RunnableLambda(lambda x: x * 2)
subtract_three = RunnableLambda(lambda x: x - 3)

# 2. Link them together cleanly using the pipe (|) operator
# This creates a reusable data pipeline
math_pipeline = add_five | multiply_by_two | subtract_three

# 3. Pass a starting value through the entire pipeline
print("--- Testing our First Custom Pipe ---")
output = math_pipeline.invoke(10)

# Execution order: (10 + 5) = 15 -> (15 * 2) = 30 -> (30 - 3) = 27
print(f"Starting Input: 10")
print(f"Final Pipeline Output: {output}")


--- Testing our First Custom Pipe ---
Starting Input: 10
Final Pipeline Output: 27


---

## 📑 Step 2: The Core Components of an LCEL Chain

Now that we understand how data moves through a pipeline, let's look at a standard production AI chain.

Every standard LangChain pipeline relies on the **Runnable Protocol**. Because Prompts, Models, and Parsers all inherit from this same background protocol, they speak the exact same data language. The output type of the component on the left perfectly matches the input type required by the component on the right.

### The Standard AI Data Flow:
```text
Input Dict {"topic": "..."}
   └──> PromptTemplate (Converts dict into a formatted text object)
           └──> ChatModel (Converts text into an AI Message object)
                   └──> OutputParser (Converts AI Message into a clean string)
```

Let's build this standard pipeline configuration.


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Prepare the standard independent components
prompt = ChatPromptTemplate.from_template("Tell me a short, one-sentence joke about {topic}.")
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()

# 2. Build the LCEL Chain
# Data seamlessly flows from Prompt -> Model -> Parser
joke_chain = prompt | model | parser

print("--- Running our First AI Chain ---")
# Invoke passes our dictionary straight into the start of the pipeline
final_joke = joke_chain.invoke({"topic": "programming"})

print(f"Resulting Output:\n{final_joke}")


--- Running our First AI Chain ---
Resulting Output:
Why do programmers prefer dark mode? Because light attracts bugs!


---

## 📑 Step 3: Inspecting Input and Output Schemas

To ensure you feel completely confident before we move on to Agents, let's verify what data types are moving between these pipes.

We can use the built-in `.input_schema.schema()` and `.output_schema.schema()` methods to inspect exactly what each component expects to receive and what it returns.


In [10]:
print("--- 🔍 Component Structural Inspection ---")

# 1. Check the Prompt requirements
# FIX: Swapped .schema() to .model_json_schema()
print(f"Prompt Input Expects:  {prompt.input_schema.model_json_schema().get('properties', {})}")
print(f"Prompt Output Returns: A Formatted Prompt Object\n")

# 2. Check the Model requirements
print(f"Model Input Expects:   A List of Message Objects or Prompt Objects")
print(f"Model Output Returns:  An AIMessage Object\n")

# 3. Check the Parser requirements
# FIX: Swapped .schema() to .model_json_schema()
print(f"Parser Input Expects:  An AIMessage Object")
print(f"Parser Output Returns: {parser.output_schema.model_json_schema().get('type')}\n")

# 4. Check the Final Combined Chain requirements
print("--- 🚀 Overall Chain Schema ---")
# FIX: Swapped .schema() to .model_json_schema()
print(f"The Chain Expects:     {joke_chain.input_schema.model_json_schema().get('properties', {})}")
print(f"The Chain Returns:     {joke_chain.output_schema.model_json_schema().get('type')}")


--- 🔍 Component Structural Inspection ---
Prompt Input Expects:  {'topic': {'title': 'Topic', 'type': 'string'}}
Prompt Output Returns: A Formatted Prompt Object

Model Input Expects:   A List of Message Objects or Prompt Objects
Model Output Returns:  An AIMessage Object

Parser Input Expects:  An AIMessage Object
Parser Output Returns: string

--- 🚀 Overall Chain Schema ---
The Chain Expects:     {'topic': {'title': 'Topic', 'type': 'string'}}
The Chain Returns:     string


## 📑 Step 4: Building Sequential Chains (Passing Data Between Prompts)

In production, you often need to string multiple AI tasks together. For example, you might use one prompt to brainstorm a concept and a second prompt to evaluate or market that concept.

### The Challenge:
Normally, you would have to capture the string output of the first model call, clean it up, create a new dictionary wrapper, and manually pass it into the second model call.

### The LCEL Solution:
By using the pipe (`|`) operator along with a simple Python dictionary conversion step, we can create a single pipeline where data flows seamlessly from the first task straight into the second task.


In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Initialize our base model and string parser
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()

# 2. Prompt A: Generate a unique startup name based on an industry variable
startup_prompt = PromptTemplate.from_template(
    "Generate a unique, catchy one-word name for a startup company in the {industry} space. Return ONLY the name."
)

# 3. Prompt B: Take that generated name and write a tagline for it
tagline_prompt = PromptTemplate.from_template(
    "Write a punchy, professional marketing slogan for a startup named '{startup_name}'."
)

# 4. Construct the Sequential Chain using LCEL
# We use a simple lambda function to map the output text of Prompt A
# into the exact dictionary key string ('startup_name') that Prompt B expects.
sequential_chain = (
    startup_prompt

    | model
    | parser
    | (lambda name: {"startup_name": name.strip()})  # Slices the text into the correct input format

    | tagline_prompt
    | model
    | parser
)

print("--- Running Sequential Chain ---")
# Kick off the chain by passing the initial industry input
final_slogan = sequential_chain.invoke({"industry": "renewable energy"})

print(f"Final Marketing Result: {final_slogan}")


--- Running Sequential Chain ---
Final Marketing Result: "Ecolux: Elevate Your Lifestyle, Sustainably."


## 📑 Step 5: Visualizing the Data Handoff with Global Logging

To truly understand how data flows through a multi-stage sequential chain, we can turn on LangChain's global `set_debug(True)` trace.

Because a sequential chain passes information from one LLM to another, the trace will reveal exactly how our inline `lambda` function catches the output of the first prompt, reshapes it, and injects it straight into the second prompt.


In [6]:
from langchain_core.globals import set_debug
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Enable global logging diagnostics
set_debug(True) #

# 2. Declare our components
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()

startup_prompt = PromptTemplate.from_template(
    "Generate a unique, catchy one-word name for a startup company in the {industry} space. Return ONLY the word."
)
tagline_prompt = PromptTemplate.from_template(
    "Write a punchy marketing slogan for a startup named '{startup_name}'."
)

# 3. Construct our multi-stage pipeline using LCEL
sequential_chain = (
    startup_prompt

    | model
    | parser
    | (lambda name: {"startup_name": name.strip()})  # Watch this step execute in logs!

    | tagline_prompt
    | model
    | parser
)

print("\n=== START OF SEQUENTIAL CHAIN LOGS ===")
# Invoke triggers the entire cascading pipeline flow
final_slogan = sequential_chain.invoke({"industry": "renewable energy"})
print("=== END OF SEQUENTIAL CHAIN LOGS ===\n")

print(f"🎉 Final Pipeline Output: {final_slogan}")

# 4. Turn debug off so subsequent workbook cells stay clean
set_debug(False)



=== START OF SEQUENTIAL CHAIN LOGS ===
[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "industry": "renewable energy"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "industry": "renewable energy"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Generate a unique, catchy one-word name for a startup company in the renewable energy space. Return ONLY the word."
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Energize",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
       

## 🏁 Bridging the Gap: Why Linear Chains Aren't Enough for Agents

Today, we built powerful, multi-step pipelines using LCEL. However, look closely at our Sequential Chain:
* It is **deterministic** and **linear**.
* Step A *always* goes to Step B, which *always* goes to Step C.
* The chain cannot stop itself, rethink its goal, or choose a different route based on a bad output.

### 🧠 Enter Autonomous Agents (Lesson 6 Preview)
An **Agent** breaks this rigid track. Instead of a hardcoded pipeline, an agent uses the LLM as a "decision engine" inside a **cyclic loop**.

In our next session, we will install **LangGraph** (LangChain's modern agent framework). We will take the Tools from Lesson 2, the Prompts from Lesson 3, the Memory from Lesson 4, and use the LCEL concepts from today to build an AI assistant that can dynamically choose its own steps until its job is complete!
